In [ ]:
import pandas as pd

sample = pd.read_csv('../data/final_200sample.csv')

In [ ]:
from openai import OpenAI
client = OpenAI()

def result(model, system_prompt, user_prompt, temp=0.55):
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=temp
    )
    return completion.choices[0].message.content

template_system = """You are simulating a response from an Original Poster (OP) in a Reddit discussion on the /r/ChangeMyView subreddit. You will be provided with the title and body of the OP's post expressing their initial view on the topic. Additionally, you will receive a top-level comment responding to the OP's post.

As the OP, your task is to respond thoughtfully to the comment, considering whether the arguments presented have changed your view. You must decide whether to award a 'delta' (Δ) based on the following criteria:

### When to Award a Delta:
- Your view has **fundamentally changed** in a meaningful way.
- A key aspect of your argument is **significantly revised** after considering the response.
- The response provides an insight that **shifts how you evaluate the topic**, even if your stance is not fully reversed.

### When Not to Award a Delta:
- You **acknowledge a strong point** but maintain your original stance.
- Your view is **only slightly adjusted** without a meaningful shift in thinking.
- You already agreed with the comment before the discussion.
- You find the argument thought-provoking but **not compelling enough to change your belief**.

### Instructions:
1. Read the **title and body** of the OP’s initial post.
2. Read the **top-level comment** responding to your post.
3. Write a response **emulating the OP’s voice**, engaging with the argument thoughtfully.
4. Decide whether to award a **delta (Δ)** based on the criteria.
   - If awarding a delta, include **'!delta' or 'Δ'** in your response.
   - If not awarding a delta, ensure these symbols are **not present**.
5. Keep your response **concise and to the point** while still addressing key arguments.
6. Aim for **3-5 sentences** unless a more detailed explanation is necessary.
7. Avoid excessive repetition or unnecessary elaboration.

---
**Title of OP's Initial Post:**  
{title_OP}

**Body of OP's Initial Post:**  
{text_OP}

"""

template_user = '''
Here is a top-level comment responding to your post and challenging your view.

**Top-Level Comment Responding to OP's Post:**  
{text_replyto}

**OP’s Response:**  
- Start your response here. Address the comment’s points, reflect on whether they have changed your view, and conclude with your decision on awarding a delta.  
- If awarding a delta, make sure to include **'!delta' or 'Δ'** in your response.
'''

model = 'gpt-4o'

results = pd.DataFrame({'id':[],
    'speaker':[],
    'conversation_id':[],
    'reply_to':[],
    'timestamp':[],
    'text':[],
    'meta':[],
    'order':[],
    'text_replyto':[],
    'text_OP':[],
    'title_OP':[],
    'has_delta':[],
    'response_LLM':[]})

for index, row in sample.iterrows():
    title_OP = row['title_OP']
    text_OP = row['text_OP']
    text_replyto = row['text_replyto']

    system_prompt = template_system.format(title_OP=title_OP, text_OP=text_OP)
    user_prompt = template_user.format(text_replyto=text_replyto)

    response_LLM = result(model, system_prompt, user_prompt)
    print(response_LLM)

    sample.at[index, 'Response_LLM'] = response_LLM

    sample.to_csv(f'data/results_{model}.csv', index=False)

Thank you for your thoughtful response! I appreciate hearing about your experiences and the perspective of someone who has moved further along in life. You make a compelling point about the continued discovery of new interests and experiences as an adult. I hadn't considered how our tastes and preferences evolve, allowing for new and exciting experiences even after childhood.

Your example of discovering a new book genre as an adult is a great illustration of how life can continue to surprise and delight us. It reminds me that while childhood is full of new experiences, adulthood has its own unique set of discoveries that can be just as fulfilling. I also hadn't fully appreciated the freedom that comes with financial independence and the ability to make choices about how to spend your time and resources, which is something I have yet to fully experience.

Regarding university life, I can see how it serves as a bridge between childhood and adulthood, offering a mix of freedom and respon

check the results

In [ ]:
res_path = "data/results_gpt-4o.csv"

res_df = pd.read_csv(res_path)

# create column for LLM delta

def has_delta(text):
    if '!delta' in text or 'Δ' in text:
        return True
    else:
        return False

res_df['LLM_delta'] = res_df['Response_LLM'].apply(lambda x: has_delta(x))

# evaluate correctness

def score(row):
    try:
        if row['LLM_delta'] == True and row['has_delta'] == True:
            return 'TP'
        if row['LLM_delta'] == False and row['has_delta'] == True:
            return 'FN'
        if row['LLM_delta'] == True and row['has_delta'] == False:
            return 'FP'
        if row['LLM_delta'] == False and row['has_delta'] == False:
            return 'TN'
    except KeyError:
        print(row)
    
res_df['result'] = res_df.apply(lambda x: score(x), axis=1)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Example DataFrame
data = {'result': ['TP', 'TN', 'FP', 'FN', 'TP', 'FP', 'TN', 'FN', 'TP', 'TN']}
df = res_df

# Count occurrences of each result type
confusion_counts = df['result'].value_counts()

# Extract TP, TN, FP, FN counts
TP = confusion_counts.get('TP', 0)
TN = confusion_counts.get('TN', 0)
FP = confusion_counts.get('FP', 0)
FN = confusion_counts.get('FN', 0)

# Calculate the metrics
accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if TP + FP > 0 else 0
recall = TP / (TP + FN) if TP + FN > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if precision + recall > 0 else 0

print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1: {f1}')

Accuracy: 0.57
Precision: 0.5813953488372093
Recall: 0.5
F1: 0.5376344086021505


In [28]:
response_LLM = result(model, template_system, template_user)